# MoSAIC-LiITA — Self-Contained Walkthrough

**MoSAIC** (**Mo**dular **S**parql **A**ssembler for **I**nterlinked **C**orpora) translates
natural-language questions into SPARQL queries against the **LiITA** (Linked Italian Language
Resources) knowledge graph.

## Resources in LiITA

| Resource | What it contains | Endpoint |
|---|---|---|
| **LiITA** | ~1.2 M Italian lemmas (LiLa) with POS, dialect variants (Sicilian, Parmigiano) | `https://liita.it/sparql` |
| **CompL-IT** | Italian lexical-semantic network: synonyms, hypernyms, hyponyms, definitions | Federated via `SERVICE` |
| **ELIta** | Emotion annotations per lemma (Gioia, Paura, Rabbia, …) | Inside LiITA graph |
| **Sentix** | Polarity values per lemma (positive/negative score) | Inside LiITA graph |

## Two Translation Modes

| Mode | How it works | When to use |
|---|---|---|
| **Deterministic** | Keyword rules select and configure pre-built SPARQL blocks. No LLM. | Predictable queries matching known patterns |
| **Agentic** | An LLM reads a tool catalogue and returns a JSON plan; the system assembles SPARQL from it. | Novel queries, multi-step reasoning, flexible combinations |

Both modes produce the same SPARQL; they differ only in *how* the plan is built.

## Pipeline Architecture

Every translation follows the same four stages:

```
NL question
    │
    ▼
[Planner / QueryAgent]  ← decides which SPARQL blocks are needed and fills their slots
    │  produces QuerySpec (list of BlockCall + SELECT/GROUP BY/aggregates)
    ▼
QuerySpec.compile()     ← topological sort: orders blocks by requires/provides dependencies
    │  produces QueryPlan (ordered list of BlockInstance)
    ▼
Assembler.assemble()    ← renders PREFIX declarations + WHERE lines into a SPARQL string
    │
    ▼
SPARQL string
```

The **Planner** (deterministic) and **QueryAgent** (agentic) both output a `QuerySpec`.
Everything downstream is identical.

## 1. Setup

In [1]:
import json
from pathlib import Path

import requests

from mosaic_liita import (
    Planner,
    Assembler,
    QueryAgent,
    make_registry,
)
from shared.llm import create_llm_client
from shared.sparql import fetch_sparql

In [2]:
# Configuration
DATA_DIR = Path("data")
ONTOLOGY_PATH = DATA_DIR / "ontology_filtered.json" 

In [3]:
# Load catalog and initialize components
with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    catalog = json.load(f)["documents"]

registry = make_registry()
planner = Planner(registry, catalog)
assembler = Assembler()

print(f"Loaded {len(catalog)} catalog entries")
print(f"Registry contains {len(registry.blocks)} blocks")

Loaded 186 catalog entries
Registry contains 20 blocks


## 2. The Block System

A `Block` is a named, reusable SPARQL fragment with explicit dependency declarations.

| Field | Type | Purpose |
|---|---|---|
| `id` | `str` | Unique name, e.g. `TRANSLATE_TO_SICILIANO` |
| `requires` | `Set[str]` | SPARQL variables that must already be bound before this block runs |
| `provides` | `Set[str]` | Variables this block binds for downstream blocks |
| `prefixes` | `Set[str]` | Prefix keys that must be declared in the query header |
| `where` | `List[str]` | SPARQL pattern lines (no surrounding `WHERE { }`) |
| `service_iri` | `str or None` | If set, wraps lines in `SERVICE <iri> { … }` (federated query) |

Slots (`{slot_name}`) are Python `str.format()` placeholders filled by the Planner or Agent.
The registry ships with ~20 predefined blocks, but the Planner can also synthesize
one-off blocks (e.g. `LIITA_BY_WRITTENREP`) for queries that need a custom filter.

In [4]:
# Inspect three representative blocks

# 1. A simple join block (no service, clear requires/provides)
print("=== JOIN_WORDREL_TO_LIITA ===")
b = registry.blocks["JOIN_WORDREL_TO_LIITA"]
print(f"  requires : {b.requires}")
print(f"  provides : {b.provides}")
print(f"  where    : {b.where}")

print()

# 2. A federated block (wraps lines in SERVICE <CompL-IT>)
print("=== COMPLIT_DEF_FILTER_BY_PATTERN ===")
b = registry.blocks["COMPLIT_DEF_FILTER_BY_PATTERN"]
print(f"  service  : {b.service_iri}")
print(f"  provides : {b.provides}")
print(f"  where (first 3 lines):")
for ln in b.where[:3]:
    print(f"    {ln}")

print()

# 3. A translation block (dialect lookup)
print("=== TRANSLATE_TO_SICILIANO ===")
b = registry.blocks["TRANSLATE_TO_SICILIANO"]
print(f"  requires : {b.requires}")
print(f"  provides : {b.provides}")
print(f"  where:")
for ln in b.where:
    print(f"    {ln}")

=== JOIN_WORDREL_TO_LIITA ===
  requires : {'?wordRel'}
  provides : {'?liitaLemma'}
  where    : ['?wordRel ontolex:canonicalForm ?liitaLemma .']

=== COMPLIT_DEF_FILTER_BY_PATTERN ===
  service  : https://klab.ilc.cnr.it/graphdb-compl-it/
  provides : {'?definition', '?itLemmaString', '?word'}
  where (first 3 lines):
    ?word a ontolex:Word ;
          lexinfo:partOfSpeech [ rdfs:label ?posLabel ] ;
          ontolex:sense ?sense ;

=== TRANSLATE_TO_SICILIANO ===
  requires : {'?liitaLemma'}
  provides : {'?sicWR', '?sicLemma'}
  where:
    ?itToSicEntry ontolex:canonicalForm ?liitaLemma ;
                 vartrans:translatableAs ?sicEntry .
    ?sicEntry ontolex:canonicalForm ?sicLemma .
    ?sicLemma dcterms:isPartOf <http://liita.it/data/id/DialettoSiciliano/lemma/LemmaBank> ;
             ontolex:writtenRep ?sicWR .


The three blocks above illustrate the spectrum:

- **JOIN_WORDREL_TO_LIITA** — a pure graph traversal: takes `?wordRel` (a CompL-IT word) and
  binds `?liitaLemma` (the corresponding LiITA lemma). No service, no slots.
- **COMPLIT_DEF_FILTER_BY_PATTERN** — runs inside the CompL-IT federated endpoint via `SERVICE`.
  Its `{lemma_filter}` and `{def_filter}` slots are filled with `FILTER(…)` fragments at planning time.
- **TRANSLATE_TO_SICILIANO** — needs `?liitaLemma` from a prior block and produces `?sicLemma`
  + `?sicWR`. It cross-joins through the dialect translation graph.

The `compile()` step topologically sorts these blocks so that every `requires` variable
is provided by an earlier block before it is used.

In [5]:
def execute_sparql(sparql: str, limit: int = 10) -> dict:
    """Execute a SPARQL query against the LiITA endpoint."""
    return fetch_sparql(sparql, limit=limit)


def display_results(data: dict):
    """Display SPARQL results in a readable format."""
    variables = data.get("head", {}).get("vars", [])
    bindings = data.get("results", {}).get("bindings", [])

    print(f"Results: {len(bindings)} rows")
    print(f"Variables: {', '.join(variables)}\n")

    for i, row in enumerate(bindings, 1):
        print(f"--- Row {i} ---")
        for var in variables:
            if var in row:
                val = row[var].get("value", "")
                print(f"  {var}: {val}")
        print()

---

## 3. Deterministic Mode

The `Planner` is a pure rule-based system: it normalises the question, detects **intent signals**
via keyword matching, then selects and configures the appropriate blocks.

| Signal detected | Blocks added |
|---|---|
| `"synonym"`, `"antonym"`, `"hyponym"`, … | `COMPLIT_SEMREL_OF_SEED_LEMMA` |
| `"definition"` + pattern word | `COMPLIT_DEF_FILTER_BY_PATTERN` |
| `"sicilian"` / `"siciliano"` | `SICILIANO_LEMMA_FILTER_…` or `TRANSLATE_TO/FROM_SICILIANO` |
| `"parmigiano"` / `"parmesan"` | `PARMIGIANO_LEMMA_FILTER_…` or `TRANSLATE_TO/FROM_PARMIGIANO` |
| `"emotion"`, `"gioia"`, `"paura"`, … | `ELITA_EMOTION_FILTER` |
| `"polarity"`, `"positive"`, `"negative"` | `SENTIX_POLARITY` |
| cross-resource joins | `JOIN_WORDREL_TO_LIITA`, `ASSERT_LIITA_LEMMA_TYPE` |

No LLM call is made. The entire translation is deterministic and sub-millisecond.

In [6]:
def translate_deterministic(question: str) -> str:
    """Translate a natural language question to SPARQL using the deterministic planner."""
    spec = planner.plan(question)
    plan = spec.compile(registry)
    sparql = assembler.assemble(plan)
    
    # Show plan details
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    print(f"\nBlocks used ({len(plan.blocks)}):")
    for i, bi in enumerate(plan.blocks, 1):
        print(f"  {i}. {bi.block.id}")
        if bi.slots:
            for k, v in bi.slots.items():
                if v:
                    print(f"     - {k}: {v}")
    
    print(f"\nSelect variables: {plan.select_vars}")
    if plan.aggregates:
        print(f"Aggregates: {plan.aggregates}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Semantic relation — synonyms of a word

Keyword `"synonym"` triggers `COMPLIT_SEMREL_OF_SEED_LEMMA` with `rel_triple` filled to the
`lexinfo:approximateSynonym` predicate (bidirectional UNION). The seed lemma `"origine"` is
quoted and injected into the `FILTER`.

In [7]:
sparql = translate_deterministic("Find synonyms of 'origine'")

Query: Find synonyms of 'origine'

Blocks used (1):
  1. COMPLIT_SEMREL_OF_SEED_LEMMA
     - seed_lemma: "origine"
     - rel_triple: { ?seedSense lexinfo:approximateSynonym ?senseRel . } UNION { ?senseRel lexinfo:approximateSynonym ?seedSense . }

Select variables: ['?wordRel']
Aggregates: {'?definitionSample': 'SAMPLE(?definition)'}

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?wordRel (SAMPLE(?definition) AS ?definitionSample)
WHERE {
  SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
    ?seedWord a ontolex:Word ;
              lexinfo:partOfSpeech [ rdfs:label ?posLabel ] ;
              ontolex:canonicalForm [ ontolex:writtenRep ?seedForm ] ;
            

In [ ]:
# Execute the query
results = execute_sparql(sparql)
display_results(results)

### Example 2: Definition lookup with prefix filter

No semantic-relation keyword is detected; `"definitions"` + `"starting with"` selects
`COMPLIT_DEF_FILTER_BY_PATTERN` with a prefix `FILTER(STRSTARTS(…))` on `?itLemmaString`.
CompL-IT uses `STRSTARTS`; LiITA uses `REGEX` — the planner picks the right syntax per endpoint.

In [9]:
sparql = translate_deterministic("Find definitions of words starting with 'ante'")

Query: Find definitions of words starting with 'ante'

Blocks used (1):
  1. COMPLIT_DEF_FILTER_BY_PATTERN
     - lemma_filter: FILTER(STRSTARTS(STR(?itLemmaString), "ante")) .

Select variables: ['?itLemmaString', '?definition']

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?itLemmaString ?definition
WHERE {
  SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
    ?word a ontolex:Word ;
          lexinfo:partOfSpeech [ rdfs:label ?posLabel ] ;
          ontolex:sense ?sense ;
          ontolex:canonicalForm ?form .
    ?form ontolex:writtenRep ?itLemmaString .
    FILTER(STRSTARTS(STR(?itLemmaString), "ante")) .
    OPTIONAL { ?sense skos:definition ?definition .

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 3: Dialect translation

`"Sicilian"` + `"ending with"` triggers `SICILIANO_LEMMA_FILTER_BY_PATTERN_AND_POS` with a
`REGEX(…, "u$")` suffix filter, then `TRANSLATE_FROM_SICILIANO` to get the Italian form.
`ASSERT_LIITA_LEMMA_TYPE` is automatically appended whenever `?liitaLemma` is in scope.

In [10]:
sparql = translate_deterministic("Find Sicilian lemmas ending with 'u' and translate them into Italian")

Query: Find Sicilian lemmas ending with 'u' and translate them into Italian

Blocks used (3):
  1. SICILIANO_LEMMA_FILTER_BY_PATTERN_AND_POS
     - wr_filter: FILTER(REGEX(STR(?sicWR), "u$")) .
  2. TRANSLATE_FROM_SICILIANO
  3. ASSERT_LIITA_LEMMA_TYPE

Select variables: ['?liitaLemma', '?itLemmaString', '?sicLemma', '?sicWR']

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX lila: <http://lila-erc.eu/ontologies/lila/>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX vartrans: <http://www.w3.org/ns/lemon/vartrans#>

SELECT ?liitaLemma ?itLemmaString ?sicLemma ?sicWR
WHERE {
    ?sicLemma a lila:Lemma .
    ?sicLemma dcterms:isPartOf <http://liita.it/data/id/DialettoSiciliano/lemma/LemmaBank> .
    
    ?sicLemma ontolex:writtenRep ?sicWR .
    FILTER(REGEX(STR(?sicWR), "u$")) .
  ?sicEntry ontolex:canonicalForm ?sicLemma .
  ?itEntry 

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 4: Multi-resource join — emotions and semantic relations

This query crosses three resources in one SPARQL:
1. **CompL-IT** (federated `SERVICE`) — finds hyponyms of *sentimento*
2. **ELIta** — filters by emotion `elita:Gioia` using `VALUES ?emotion { … }` (IRI-based, avoids Virtuoso string-function permission errors)
3. **Sentix** — retrieves polarity scores

The planner detects `"hyponym"` → `COMPLIT_SEMREL_OF_SEED_LEMMA`, `"emotion"` / `"gioia"` →
`ELITA_EMOTION_FILTER`, and appends `SENTIX_POLARITY` because positive-emotion context is present.

In [11]:
sparql = translate_deterministic(
    "Find Italian words that express positive emotions ('gioia') and are hyponyms of 'sentimento'"
)

Query: Find Italian words that express positive emotions ('gioia') and are hyponyms of 'sentimento'

Blocks used (5):
  1. COMPLIT_SEMREL_OF_SEED_LEMMA
     - seed_lemma: "sentimento"
     - rel_triple: ?senseRel lexinfo:hyponym ?seedSense .
  2. JOIN_WORDREL_TO_LIITA
  3. ASSERT_LIITA_LEMMA_TYPE
  4. ELITA_EMOTION_FILTER
     - emotion_filter_clause: VALUES ?emotion { elita:Gioia }
  5. SENTIX_POLARITY

Select variables: ['?liitaLemma', '?itLemmaString']
Aggregates: {'?definitionSample': 'SAMPLE(?definition)', '?emotions': 'GROUP_CONCAT(DISTINCT ?emotionLabel; SEPARATOR=", ")', '?polarityLabels': 'GROUP_CONCAT(DISTINCT ?polarityLabel; SEPARATOR=", ")', '?polarityValues': 'GROUP_CONCAT(DISTINCT ?polarityValue; SEPARATOR=", ")'}

------------------------------------------------------------
Generated SPARQL:
------------------------------------------------------------
PREFIX elita: <http://w3id.org/elita/>
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX lila: <http:

In [ ]:
results = execute_sparql(sparql)
display_results(results)

---

## 4. Agentic Mode

### How it actually works

Despite the name, the agentic mode is **not** a multi-step agent loop. It is a **single LLM call**
that returns a structured JSON plan, which the system then assembles without further LLM calls.

```
NL question
    │
    ▼
One LLM call
  ┌──────────────────────────────────────────────────────┐
  │ System prompt: catalogue of available "tools"        │
  │   (one entry per Block: name, description, params)   │
  │                                                      │
  │ User message: the NL question                        │
  │                                                      │
  │ LLM responds with JSON:                              │
  │   {                                                  │
  │     "reasoning": "...",                              │
  │     "steps": [                                       │
  │       {"tool": "find_liita_lemmas_by_pattern",       │
  │        "params": {"pattern_type": "exact",           │
  │                   "pattern_text": "uccello"}},       │
  │       {"tool": "translate_to_parmigiano", ...}       │
  │     ],                                               │
  │     "output_vars": ["?parWR", "?wr"],                │
  │     "aggregation": null                              │
  │   }                                                  │
  └──────────────────────────────────────────────────────┘
    │
    ▼
plan_to_queryspec()  ← converts each JSON step into a BlockCall
    │
    ▼
QuerySpec → compile() → Assembler → SPARQL   (same pipeline as deterministic)
```

The LLM **never executes** any tool. It only *plans*. The JSON is validated, then converted
to `BlockCall`s by `plan_to_queryspec()`.

### When to use agentic mode

- Queries combining resources in ways the deterministic rules don't cover
- Freer natural language (less structured phrasing)
- Novel combinations of blocks not seen during planner development

**Limitation:** LLM output is non-deterministic — the same query may produce slightly different
plans across runs. The deterministic planner is faster and fully reproducible.

In [12]:
# Inspect the tool catalogue the LLM receives in its system prompt
from mosaic_liita.agent import AGENT_TOOLS

print(f"Tool catalogue: {len(AGENT_TOOLS)} tools")
print()
for i, (name, spec) in enumerate(AGENT_TOOLS.items()):
    if i >= 5:
        break
    desc = spec.get("description", "")
    print(f"  [{name}]")
    print(f"    {desc[:90]}{'...' if len(desc) > 90 else ''}")
    if spec.get("parameters"):
        print(f"    params: {list(spec['parameters'].keys())}")
    print()
if len(AGENT_TOOLS) > 5:
    print(f"  ... and {len(AGENT_TOOLS) - 5} more tools")

Tool catalogue: 18 tools

  [find_semantic_relations]
    Find words that have a semantic relation (hyponym, hypernym, synonym, antonym, meronym, ho...
    params: ['seed_word', 'relation_type', 'pos_filter']

  [find_definitions_by_pattern]
    Find CompL-IT words and definitions where the lemma OR definition matches a pattern (start...
    params: ['pattern_type', 'pattern_text', 'apply_to', 'pos_filter']

  [find_liita_lemmas_by_pattern]
    Find Italian lemmas in LiITA that match a pattern (starts with, ends with, contains). Use ...
    params: ['pattern_type', 'pattern_text', 'pos_filter']

  [find_sicilian_lemmas_by_pattern]
    Find Sicilian dialect lemmas that match a pattern. Use this when the query specifically as...
    params: ['pattern_type', 'pattern_text']

  [find_parmigiano_lemmas_by_pattern]
    Find Parmigiano dialect lemmas that match a pattern. Use this when the query specifically ...
    params: ['pattern_type', 'pattern_text']

  ... and 13 more tools


In [16]:
# LLM Configuration
# Choose your provider: "mistral", "anthropic", "openai", "gemini", "ollama"

LLM_PROVIDER = ""  # Change this to your provider
LLM_API_KEY = ""         # Not needed for Ollama
LLM_MODEL = ""   # Or leave empty for default

# Default models per provider:
# - mistral: "mistral-large-latest"
# - anthropic: "claude-sonnet-4-20250514"
# - openai: "gpt-4.1-mini"
# - gemini: "gemini-1.5-pro"
# - ollama: "llama3.1"

In [17]:
# Initialize LLM client and agent
llm_client = create_llm_client(
    provider=LLM_PROVIDER,
    api_key=LLM_API_KEY if LLM_API_KEY else None,
    model=LLM_MODEL,
)

agent = QueryAgent(registry, catalog, llm_client)
print(f"Agent initialized with {LLM_PROVIDER} ({LLM_MODEL or 'default model'})")

Agent initialized with ollama (default model)


In [18]:
# Print the exact prompts sent to the LLM for an example query
example_q = "Find synonyms of 'origine' whose Sicilian translation starts with 'm'"

print("=" * 70)
print("SYSTEM PROMPT")
print("=" * 70)
print(agent._build_system_prompt())

print()
print("=" * 70)
print(f"USER PROMPT  (question: {example_q})")
print("=" * 70)
print(agent._build_prompt(example_q))

SYSTEM PROMPT
You are a query planner for the LiITA Italian linguistic database. Your job is to decompose natural language queries into a structured plan using the available tools.

IMPORTANT RULES:
1. Output ONLY valid JSON matching the schema below
2. Only use tools from the provided list
3. Ensure variable dependencies are satisfied (check "requires" for each tool)
4. Be minimal - don't add unnecessary tools
5. Never invent new tools or parameters

When a query needs:
- Translations: First get ?liitaLemma (via join_to_liita or bind_lemma_to_liita), then use translate_to_*
- Sentiment: First get ?liitaLemma, then use get_sentiment
- Emotions: First get ?liitaLemma, then use get_emotions
- Additional filtering: Use filter_variable after the main query tools

How to get ?liitaLemma:
- If the query starts from a specific word pattern: use find_liita_lemmas_by_pattern (pattern_type/pattern_text) then bind_lemma_to_liita
- If the query filters Italian words by emotion/sentiment (NO specif

In [ ]:
def translate_agentic(question: str) -> str:
    """Translate a natural language question to SPARQL using the agentic planner."""
    sparql, agent_plan, query_spec = agent.translate(question)
    
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    
    print(f"\nAgent Reasoning:")
    print(f"  {agent_plan.reasoning}")
    
    print(f"\nTool Calls ({len(agent_plan.steps)}):")
    for i, step in enumerate(agent_plan.steps, 1):
        print(f"  {i}. {step.tool}")
        if step.params:
            for k, v in step.params.items():
                print(f"     - {k}: {v}")
    
    print(f"\nOutput variables: {agent_plan.output_vars}")
    
    if agent_plan.aggregation:
        print(f"Aggregation: {agent_plan.aggregation}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Chained semantic + dialect filter

This query requires the agent to chain `find_semantic_relations` → `join_wordrel_to_liita`
→ `translate_to_sicilian` and add a suffix filter on `?sicWR`. The deterministic planner
cannot handle the combined "synonym + dialect filter" in one pass.

In [ ]:
sparql = translate_agentic("Find synonyms of 'origine' whose Sicilian translation starts with 'm'")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 2: Aggregation — counting

The agent must recognise that `"how many"` requires a `COUNT(DISTINCT …)` aggregate.
It uses `find_sicilian_lemmas_by_pattern` with no pattern filter (match-all), then sets
`aggregation.count_variable` in its JSON response.

In [ ]:
sparql = translate_agentic("How many lemmas are present in the Sicilian lexicon?")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 3: Multi-dialect enrichment

This query asks for both Parmigiano and Sicilian translations of CompL-IT words whose
definition starts with a given prefix. The agent chains `find_definitions_by_pattern`
→ `join_word_to_liita` → `translate_to_parmigiano` + `translate_to_sicilian` in parallel.

In [ ]:
sparql = translate_agentic(
    "Find CompL-IT definitions starting with 'uccello' and show Parmigiano and Sicilian translations"
)

In [ ]:
results = execute_sparql(sparql)
display_results(results)

---

## 5. Comparison

Run the same query through both modes to see how their plans differ.

**What to look for:**
- Deterministic: fewer blocks, very targeted slot values, sub-millisecond planning
- Agentic: may add extra enrichment blocks or use a different aggregation strategy
- Both should return equivalent result sets (modulo ordering)

The comparison is most interesting for queries that sit on the boundary — queries the
deterministic planner handles but where the agentic mode takes a different route.

In [ ]:
test_query = "Find synonyms of 'origine' with their Sicilian translations"

print("DETERMINISTIC MODE")
print("=" * 60)
sparql_det = translate_deterministic(test_query)

In [ ]:
print("\nAGENTIC MODE")
print("=" * 60)
sparql_agent = translate_agentic(test_query)

In [ ]:
# Compare results
print("Deterministic results:")
display_results(execute_sparql(sparql_det))

print("\nAgentic results:")
display_results(execute_sparql(sparql_agent))